# Упражнения <a id='tasks'></a>

Необходимо изучить библиотеку [tslearn](https://tslearn.readthedocs.io/en/stable/).

1. Взять выбранные для лабы 14 набор(ы) данных. 

3. Попробовать все методы классификации и регрессии, описанные в блоноте 14 aeon:
    Distance-based 
    Свертки (модели семейства Rocket и Hydra)
    Feature-based
    Deep Learning

    Для каждого подхода обучть не менее 3х разных моделей для классификации и регрессии.
    
     
    

In [1]:
from aeon.datasets import load_from_ts_file
from tslearn.utils import to_time_series_dataset
from sklearn.preprocessing import FunctionTransformer

from tslearn.svm import TimeSeriesSVC

from tslearn.neighbors import KNeighborsTimeSeriesClassifier
from tslearn.metrics import dtw, soft_dtw
from sklearn.metrics import accuracy_score

X_train, y_train = load_from_ts_file("./dataset/KDD_MTSC_TRAIN.ts")
X_test, y_test = load_from_ts_file("./dataset/KDD_MTSC_TEST.ts")

Xr_train, yr_train = load_from_ts_file("./dataset/KDD_MTSER_TRAIN.ts")
Xr_test, yr_test = load_from_ts_file("./dataset/KDD_MTSER_TEST.ts")

X_train = X_train.transpose(0, 2, 1)
X_test = X_test.transpose(0, 2, 1)

Xr_train = Xr_train.transpose(0, 2, 1)
Xr_test = Xr_test.transpose(0, 2, 1)

print("Train:", X_train.shape)

Train: (40, 100, 4)


In [2]:
# 1. KNN + DTW

knn_dtw = KNeighborsTimeSeriesClassifier(n_neighbors=3, metric="dtw")
knn_dtw.fit(X_train, y_train)
pred = knn_dtw.predict(X_test)

print("KNN + DTW:", accuracy_score(y_test, pred))

# 2. KNN + SoftDTW

knn_soft = KNeighborsTimeSeriesClassifier(n_neighbors=3, metric="softdtw")
knn_soft.fit(X_train, y_train)
pred = knn_soft.predict(X_test)

print("KNN + SoftDTW:", accuracy_score(y_test, pred))

svc = TimeSeriesSVC(kernel="gak")  # Global Alignment Kernel
svc.fit(X_train, y_train)
pred = svc.predict(X_test)

print("SVM (GAK):", accuracy_score(y_test, pred))

KNN + DTW: 0.625
KNN + SoftDTW: 0.525


C:\Users\zuben\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM (GAK): 0.4


In [3]:
from tslearn.neighbors import KNeighborsTimeSeriesRegressor
from sklearn.metrics import mean_squared_error

# 1. KNN + DTW

knn_reg = KNeighborsTimeSeriesRegressor(n_neighbors=3, metric="dtw")
knn_reg.fit(Xr_train, yr_train)
pred = knn_reg.predict(Xr_test)

print("Reg DTW:", mean_squared_error(yr_test, pred))

# 2. KNN + SoftDTW

knn_reg = KNeighborsTimeSeriesRegressor(n_neighbors=3, metric="softdtw")
knn_reg.fit(Xr_train, yr_train)
pred = knn_reg.predict(Xr_test)

print("Reg SoftDTW:", mean_squared_error(yr_test, pred))

Reg DTW: 0.8680719196007588
Reg SoftDTW: 0.8680719196007588


In [4]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from tslearn.preprocessing import TimeSeriesScalerMeanVariance

# flatten helper
def flatten(X):
    return X.reshape(X.shape[0], -1)

pipe = Pipeline([
    ("scale", TimeSeriesScalerMeanVariance()),
    ("flatten", FunctionTransformer(flatten)),
    ("clf", RandomForestClassifier(n_estimators=100))
])

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

print("Feature RF:", accuracy_score(y_test, pred))

Feature RF: 0.55


In [5]:
from tslearn.clustering import TimeSeriesKMeans
from tslearn.metrics import cdist_dtw
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

n_clusters = 10

kmeans = TimeSeriesKMeans(n_clusters=n_clusters, metric="dtw", random_state=42)
kmeans.fit(X_train)

shapelets = kmeans.cluster_centers_

X_train_feat = cdist_dtw(X_train, shapelets)
X_test_feat = cdist_dtw(X_test, shapelets)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_feat, y_train)

pred = clf.predict(X_test_feat)

print("Shape-based (via clustering):", accuracy_score(y_test, pred))

Shape-based (via clustering): 0.825


In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

Xr_flat = Xr_train.reshape(Xr_train.shape[0], -1)
Xr_test_flat = Xr_test.reshape(Xr_test.shape[0], -1)

rf = RandomForestRegressor(n_estimators=100)
rf.fit(Xr_flat, yr_train)

pred = rf.predict(Xr_test_flat)

print("RF Regression:", mean_squared_error(yr_test, pred))

RF Regression: 0.950810845734146
